# Solar Filament Segmentation Challenge 2026 - reproduce the submission

This notebook regenerates the submitted predictions from the **test images** and the **trained weights**.

Pipeline: image -> 25 overlapping 512 px tiles (stride 384) -> U-Net/ResNet34 -> Gaussian-blended probability map
-> threshold 0.5 -> connected components (pieces < 120 px dropped; nearby pieces grouped with a closing of k=31,
pixels unchanged) -> COCO RLE -> `submission.csv`.

**Before you run:** (1) download the competition test images from the Kaggle *Data* tab and put them, flat, into `test_images/`;
(2) a GPU is recommended (Colab T4 is enough). All settings live in `src/filament/config.py`.

*Verified: run on a Colab T4 GPU (about 5 minutes for the 180 images) it regenerated the submitted file - 1565 of 1566 RLE strings identical, the remaining mask has IoU 0.9983 with the submitted one. See README, section "How to verify".*

In [ ]:
# 1) install dependencies (Colab: if pip asks to restart the runtime, restart it and run again)
%pip install -q -r requirements.txt

In [ ]:
# 2) paths and settings
import os, sys, urllib.request
sys.path.insert(0, os.path.abspath("src"))

IMAGES_DIR = "test_images"                   # 180 test images, flat folder

# Weights of the two submitted models (GitHub Release v1.0, downloaded automatically if the files are missing).
MODELS = {
    "modelA_baseline": {"path": "unet_resnet34_baseline_ep18.pth",   "url": "https://github.com/lovephetPTL/filament-seg-2026/releases/download/v1.0/unet_resnet34_baseline_ep18.pth", "out": "submission_modelA_closing31.csv"},
    "modelB_v3_control": {"path": "unet_resnet34_v3_control_ep5.pth", "url": "https://github.com/lovephetPTL/filament-seg-2026/releases/download/v1.0/unet_resnet34_v3_control_ep5.pth", "out": "submission_modelB_closing31.csv"},
}

assert os.path.isdir(IMAGES_DIR), "Put the Kaggle test images into test_images/ first"
for name, m in MODELS.items():
    if not os.path.exists(m["path"]):
        assert m["url"], f"Download {m[path]} from the GitHub Release page and place it next to this notebook"
        urllib.request.urlretrieve(m["url"], m["path"])
    print(name, round(os.path.getsize(m["path"]) / 1e6, 1), "MB")
print("images:", len(os.listdir(IMAGES_DIR)))

In [ ]:
# 3) load the model helper and device
import torch
from filament.model import load_checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# 4) FINAL files: closing k=31 used only for grouping (Model A public 0.32, Model B public 0.33)
from filament.pipeline import make_submission
reports = {}
for name, m in MODELS.items():
    model = load_checkpoint(m["path"], device)
    reports[name] = make_submission(IMAGES_DIR, model, device, m["out"], closing_kernel=31)
    del model
reports

In [ ]:
# 5) (optional) BASELINE file without closing (public 0.31): plain connected components
# model = load_checkpoint(MODELS["modelA_baseline"]["path"], device)
# make_submission(IMAGES_DIR, model, device, "submission_baseline_raw.csv", closing_kernel=None)

In [ ]:
# 6) (optional) compare with the files that were uploaded to Kaggle (folder submission/ of this repository; skip this step if the folder is missing)
# !python scripts/compare_submissions.py submission/submission_baseline_closing31.csv submission_modelA_closing31.csv
# !python scripts/compare_submissions.py submission/submission_v3_control_ep5_closing31.csv submission_modelB_closing31.csv

### What to expect
* 180 test images, 2 of them without any predicted filament. Model A: 1,566 rows (about 8.7 filaments per image); Model B: 1,528 rows (about 8.49 per image).
* The self-check dictionary must show `rows_failing_rle_roundtrip = 0` and `images_with_overlapping_masks = 0`.
* Small floating-point differences between GPUs can change a few boundary pixels; step 6 reports how many RLE strings are identical.